# October Modeling Split Plan | Provisional Review

This notebook explores the previously proposed October modeling design before bringing it to the team. It uses all 188,318 eligible claims in a reproducible 70/15/15 claim-level split with random seed 2026.

The partitions are training, validation, and a reserved internal test partition. The test partition is checked for membership and row count only; its targets are not inspected and no test performance is computed.

The initial comparison is predeclared as: training-mean baseline, training-median comparator, one linear model, and one nonlinear or tree-based model. Validation MAE in original loss units is used for model comparison. A favorable validation score is not a reason to try another seed.

## 1. Imports and source data

In [1]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

def find_repo_root():
    current = Path.cwd().resolve()
    for path in [current, *current.parents]:
        if (path / 'data' / 'allstate_claims_data.csv').exists():
            return path
    raise FileNotFoundError('Repository root not found')

ROOT = find_repo_root()
SOURCE_PATH = ROOT / 'data' / 'allstate_claims_data.csv'
OUTPUT_DIR = ROOT / 'notebooks' / 'jonathan-deng' / 'october_split_outputs'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(SOURCE_PATH)
id_column = 'id'
target_column = 'loss'
cat_columns = [f'cat{i}' for i in range(1, 117)]
cont_columns = [f'cont{i}' for i in range(1, 15)]
feature_columns = cat_columns + cont_columns

print('Source:', SOURCE_PATH.relative_to(ROOT))
print('Rows:', len(df))
print('Columns:', len(df.columns))

Source: data/allstate_claims_data.csv
Rows: 188318
Columns: 132


## 2. Eligibility rules and frozen split settings

In [2]:
SEED = 2026
TRAIN_SHARE = 0.70
VALIDATION_SHARE = 0.15
TEST_SHARE = 0.15
PARTITIONS = {'train', 'validation', 'test'}

required_columns = [id_column, *feature_columns, target_column]
assert df.columns.tolist() == required_columns
assert df[id_column].is_unique
assert df[target_column].notna().all()
assert np.isfinite(df[target_column]).all()
assert (df[target_column] > 0).all()

eligible = df.loc[
    df[id_column].notna()
    & df[target_column].notna()
    & np.isfinite(df[target_column])
    & (df[target_column] > 0)
].copy()

assert len(eligible) == 188_318
assert eligible[id_column].is_unique

print('Eligibility checks: PASS')
print('Eligible claims:', len(eligible))
print('Split:', f'{TRAIN_SHARE:.0%}/{VALIDATION_SHARE:.0%}/{TEST_SHARE:.0%}')
print('Seed:', SEED)

Eligibility checks: PASS
Eligible claims: 188318
Split: 70%/15%/15%
Seed: 2026


## 3. Create and verify the ID-keyed partition manifest

The manifest is keyed by `id`. Every eligible claim must occur exactly once, and no claim may be omitted or assigned to multiple partitions.

In [3]:
rng = np.random.default_rng(SEED)
shuffled_ids = eligible[id_column].to_numpy().copy()
rng.shuffle(shuffled_ids)

train_count = int(round(len(shuffled_ids) * TRAIN_SHARE))
validation_count = int(round(len(shuffled_ids) * VALIDATION_SHARE))

partition_manifest = pd.DataFrame({id_column: shuffled_ids})
partition_manifest['partition'] = 'test'
partition_manifest.loc[:train_count - 1, 'partition'] = 'train'
partition_manifest.loc[
    train_count:train_count + validation_count - 1,
    'partition',
] = 'validation'
partition_manifest = partition_manifest.sort_values(id_column).reset_index(drop=True)

assert partition_manifest[id_column].is_unique
assert len(partition_manifest) == len(eligible)
assert set(partition_manifest[id_column]) == set(eligible[id_column])
assert set(partition_manifest['partition']) == PARTITIONS
assert partition_manifest.groupby(id_column).size().eq(1).all()

manifest_path = OUTPUT_DIR / 'provisional_partition_manifest.csv'
partition_manifest.to_csv(manifest_path, index=False)

partition_counts = partition_manifest['partition'].value_counts().sort_index()
print('Manifest checks: PASS')
display(partition_counts.to_frame('claims'))
print('Saved:', manifest_path.relative_to(ROOT))

Manifest checks: PASS


,claims
partition,
test,28247
train,131823
validation,28248


Saved: notebooks/jonathan-deng/october_split_outputs/provisional_partition_manifest.csv


## 4. Training and validation evidence

The test partition is not joined to the target-bearing data in this section. Training and validation summaries are used to check whether the split is reasonable before model comparison.

In [4]:
assigned = eligible[[id_column, target_column, *cat_columns]].merge(
    partition_manifest,
    on=id_column,
    how='left',
    validate='one_to_one',
)

assert assigned['partition'].notna().all()
train = assigned.loc[assigned['partition'] == 'train'].copy()
validation = assigned.loc[assigned['partition'] == 'validation'].copy()

target_summary = pd.DataFrame({
    'partition': ['train', 'validation'],
    'rows': [len(train), len(validation)],
    'mean_loss': [train[target_column].mean(), validation[target_column].mean()],
    'median_loss': [train[target_column].median(), validation[target_column].median()],
    'q95_loss': [train[target_column].quantile(0.95), validation[target_column].quantile(0.95)],
    'max_loss': [train[target_column].max(), validation[target_column].max()],
})

high_loss_threshold = train[target_column].quantile(0.95)
high_loss_support = pd.DataFrame({
    'partition': ['train', 'validation'],
    'threshold_source': 'training 95th percentile',
    'threshold': high_loss_threshold,
    'rows_at_or_above_threshold': [
        int((train[target_column] >= high_loss_threshold).sum()),
        int((validation[target_column] >= high_loss_threshold).sum()),
    ],
})

target_summary.to_csv(OUTPUT_DIR / 'train_validation_target_summary.csv', index=False)
high_loss_support.to_csv(OUTPUT_DIR / 'train_validation_high_loss_support.csv', index=False)
display(target_summary)
display(high_loss_support)

,partition,rows,mean_loss,median_loss,q95_loss,max_loss
0,train,131823,3038.626059,2115.560,8533.1640,121012.25
1,validation,28248,3033.989415,2130.625,8394.6975,49549.85


,partition,threshold_source,threshold,rows_at_or_above_threshold
0,train,training 95th percentile,8533.164,6592
1,validation,training 95th percentile,8533.164,1338


## 5. Categorical level coverage

In [5]:
coverage_rows = []
for column in cat_columns:
    train_levels = set(train[column].dropna().unique())
    validation_levels = set(validation[column].dropna().unique())
    unseen_validation_levels = validation_levels - train_levels
    coverage_rows.append({
        'field': column,
        'training_levels': len(train_levels),
        'validation_levels': len(validation_levels),
        'unseen_validation_levels': len(unseen_validation_levels),
        'validation_levels_seen_in_training': len(validation_levels & train_levels),
    })

categorical_coverage = pd.DataFrame(coverage_rows)
categorical_coverage.to_csv(OUTPUT_DIR / 'train_validation_categorical_coverage.csv', index=False)

print('Fields with unseen validation levels:', int((categorical_coverage['unseen_validation_levels'] > 0).sum()))
display(categorical_coverage.sort_values('unseen_validation_levels', ascending=False).head(15))

Fields with unseen validation levels: 3


,field,training_levels,validation_levels,unseen_validation_levels,validation_levels_seen_in_training
115,cat116,315,237,5,232
109,cat110,127,107,1,106
104,cat105,19,16,1,15
72,cat73,3,3,0,3
84,cat85,4,4,0,4
83,cat84,4,4,0,4
82,cat83,4,4,0,4
81,cat82,4,4,0,4
80,cat81,4,4,0,4
79,cat80,4,4,0,4


## 6. Required constant baselines

Both baselines are fitted using training targets only. They are evaluated on validation targets in original loss units.

In [6]:
def mean_absolute_error(actual, predicted):
    return np.abs(actual - predicted).mean()

training_mean = train[target_column].mean()
training_median = train[target_column].median()

baseline_results = pd.DataFrame({
    'model': ['training_mean_constant', 'training_median_constant'],
    'training_value': [training_mean, training_median],
    'validation_mae_original_units': [
        mean_absolute_error(validation[target_column], training_mean),
        mean_absolute_error(validation[target_column], training_median),
    ],
    'validation_rows': len(validation),
})

baseline_results.to_csv(OUTPUT_DIR / 'constant_baseline_results.csv', index=False)
display(baseline_results)

,model,training_value,validation_mae_original_units,validation_rows
0,training_mean_constant,3038.626059,1949.857660,28248
1,training_median_constant,2115.560000,1796.557249,28248


## 7. Provisional decision record

This record is for personal review. It does not update the team's official October manifest. The reserved test partition remains excluded from fitting, tuning, and selection, and is intended for one November final evaluation.

In [7]:
source_sha256 = hashlib.sha256(SOURCE_PATH.read_bytes()).hexdigest()
decision = {
    'decision_id': 'OCT-SPLIT-PROVISIONAL-JD-001',
    'status': 'provisional_personal_review',
    'modeling_data_version': 'OCT-MD-001',
    'eligible_claims': len(eligible),
    'split_method': 'random permutation of unique claim ids',
    'proportions': {'train': TRAIN_SHARE, 'validation': VALIDATION_SHARE, 'test': TEST_SHARE},
    'actual_counts': partition_counts.to_dict(),
    'random_seed': SEED,
    'cross_validation': 'not used initially; future folds must remain inside training',
    'validation_use': 'compare predeclared models using MAE in original loss units',
    'test_use': 'reserved for one November final evaluation; test targets remain unused in October',
    'initial_comparison': [
        'training-mean constant baseline',
        'training-median constant comparator',
        'one linear regression model',
        'one nonlinear/tree-based model',
    ],
    'stopping_rule': 'stop after all predeclared candidates are evaluated; do not seed-hunt',
    'split_change_rule': 'change only for documented technical or integrity failure',
    'source_sha256': source_sha256,
}

decision_path = OUTPUT_DIR / 'provisional_split_decision.json'
with decision_path.open('w') as handle:
    json.dump(decision, handle, indent=2)

print('Provisional decision saved:', decision_path.relative_to(ROOT))
print('No official team artifacts were changed.')

Provisional decision saved: notebooks/jonathan-deng/october_split_outputs/provisional_split_decision.json
No official team artifacts were changed.
